# PED Estimation

Using the file, we will estimate the PED using **historical data** and prices by transactions. As the end product, I would envision having estimated PED for different transactions channels, segmented by i) Special Occasion Elasticity, ii) Normal Occasion Elasticity and iii) Optimistic (Low Elasticity), Medium, and Pessimistic (High Elasticity). 

Through this analysis, we want to answer the question: What is the break-even elasticity? In other words, for each elasticity estimated, what is the maximum price increase we can impose for each transaction channel without losing number of visitors. 

**Steps:**
1. I will first construct a base model - without considering holidays and special occasions, to see if we can estimate the average PED for each transaction. 
- Since the price is static historically, this creates difficulty in modeling PED for individual transaction.
- However, we can use proxy for price, eg, the difference in price across different transaction channel, to see if there is a price change, how does it nudge individuals to choose other transaction channels.
- That might be the best plan we can do to estimate PED
- Using ratio, we can isolate the effect of special occasion, since during holidays, sales of all platform are all increasing, and the ratio might be staying the same. 

2. For now, I will left out transaction channel where 0 revenue/unit prices is recorded, since we cannot estimate PED from RM 0 unit price. 

3. Then, I will segment the data according to holiday/special occasions VS non-special occasions. To simplify our analysis, we would just consider the binary case (holiday yes VS holiday no). Once the estimation of PED is possible, we will expand the model to consider more specific cases of holiday. 

In [6]:
# Import necessary libraries
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime
import statsmodels.api as sm

## List of variables

1. Environment: Specify whether the environment is Sunway Lagoon or Lost World
- WHEN Environment = 1 THEN 'Sunway Lagoon'
- WHEN Environment = 2 THEN 'Lost World'

2. Transaction data
- Complete_QTY: Total Visitors
- OriginalAmount * Complete_QTY: Total Revenue

3. Park: Revenue Grouping (whether you are online, walk-in, bulk buying, etc)
- We should understand the different unique values available

4. TranDate: Transaction Date
- According to our EDA, we only focus on the data from 2023 to 2024 to train our model, skipping earlier years that is affected by COVID-19.

## Loading dataset

In [11]:
# Connect to the database
con = duckdb.connect("theme_park_data.db")

# Extract the necessary information for EDA
# Transactions data, channels (revenue grouping), and type of park/environment
# Focus from 2023 to 2024

df_channel_park = con.execute("""
SELECT
    TranDate AS Transaction_Date,
    TRIM(Park) as Grouping,
    CASE
        WHEN Environment = 1 THEN 'Sunway Lagoon'
        WHEN Environment = 2 THEN 'Lost World'
    END AS Park,
    OriginalAmount AS Unit_Price,
    Complete_QTY AS No_of_Visitors,
    (OriginalAmount * Complete_QTY) AS Total_Revenue
FROM theme_park_database
WHERE Environment IN (1, 2) 
    AND Grouping IS NOT NULL
    AND TranDate >= TIMESTAMP '2023-01-01 00:00:00'
    AND TranDate < TIMESTAMP '2025-01-01 00:00:00'
""").df()

df_channel_park.head()

# Clean the 'Grouping' column directly in the dataframe
# .strip() removes spaces, .title() converts 'walk in' to 'Walk In'
df_channel_park['Grouping'] = df_channel_park['Grouping'].str.strip().str.title()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [12]:
df_channel_park.head()

,Transaction_Date,Grouping,Park,Unit_Price,No_of_Visitors,Total_Revenue
0,2023-11-12 08:16:30,Group Function,Sunway Lagoon,0.0,0.0,0.0
1,2023-04-09 01:29:29,Tour / Hotel / Presold,Sunway Lagoon,0.0,0.0,0.0
2,2024-03-16 12:27:25,Membership,Lost World,0.0,0.0,0.0
3,2024-02-15 19:43:14,Hotspring,Lost World,12.0,1.0,12.0
4,2024-11-22 06:56:24,Hotspring,Lost World,0.0,0.0,0.0


In [16]:
def estimate_park_specific_ped(df):
    df = df.copy()
    
    # 1. Normalize Date (Strip Time)
    df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date']).dt.date
    
    # 2. Filter: Only market-driven channels (Remove Staff/Inter-Company/FOC)
    # These are not price-sensitive and bias the model
    excluded_groups = ['Staff', 'Inter-Company', 'Foc / Complimentary', 'Membership']
    df = df[~df['Grouping'].isin(excluded_groups)]
    df = df[(df['No_of_Visitors'] > 0) & (df['Total_Revenue'] > 0)]
        
    # 3. Aggregation: Daily totals per Park and Channel
    daily_agg = df.groupby(['Transaction_Date', 'Park', 'Grouping']).agg({
        'No_of_Visitors': 'sum',
        'Total_Revenue': 'sum'
    }).reset_index()

    daily_agg['Unit_Price'] = daily_agg['Total_Revenue'] / daily_agg['No_of_Visitors']
    
    # 4. Create the Benchmark (Walk-In) per Park
    walkin_ref = daily_agg[daily_agg['Grouping'] == 'Walk In'].copy()
    walkin_ref = walkin_ref.rename(columns={
        'Unit_Price': 'P_Base',
        'No_of_Visitors': 'Q_Base'
    })[['Transaction_Date', 'Park', 'P_Base', 'Q_Base']]

    # 5. Join and Calculate Ratios
    merged = pd.merge(daily_agg, walkin_ref, on=['Transaction_Date', 'Park'])
    
    processed = merged[
        (merged['Grouping'] != 'Walk In') & 
        (merged['Unit_Price'] > 0) & 
        (merged['P_Base'] > 0)
    ].copy()
    
    # Log Ratios for Elasticity
    processed['ln_Price_Ratio'] = np.log(processed['Unit_Price'] / processed['P_Base'])
    processed['ln_Qty_Ratio'] = np.log(processed['No_of_Visitors'] / processed['Q_Base'])

    # 6. NESTED LOOP: Estimate PED for each Channel WITHIN each Park
    results = []
    unique_parks = processed['Park'].unique()
    
    for park in unique_parks:
        park_data = processed[processed['Park'] == park]
        unique_channels = park_data['Grouping'].unique()
        
        for channel in unique_channels:
            chan_data = park_data[park_data['Grouping'] == channel].copy()
            chan_data = chan_data.replace([np.inf, -np.inf], np.nan).dropna(subset=['ln_Price_Ratio', 'ln_Qty_Ratio'])
            
            # Need a decent sample size for daily regression
            if len(chan_data) < 20: 
                continue
                
            try:
                X = sm.add_constant(chan_data['ln_Price_Ratio'])
                y = chan_data['ln_Qty_Ratio']
                
                model = sm.OLS(y, X).fit()
                
                results.append({
                    'Park': park,
                    'Transaction_Channel': channel,
                    'Relative_PED': model.params.iloc[1],
                    'R_Squared': model.rsquared,
                    'P_Value': model.pvalues.iloc[1],
                    'Sample_Size': len(chan_data)
                })
            except:
                continue

    return pd.DataFrame(results).sort_values(by=['Park', 'Relative_PED'])

# --- Execution ---
ped_results = estimate_park_specific_ped(df_channel_park)
print(ped_results)

            Park     Transaction_Channel  Relative_PED  R_Squared  \
0  Sunway Lagoon          Group Function     -1.005302   0.054022   
2  Sunway Lagoon  Tour / Hotel / Presold     -0.445153   0.011903   
1  Sunway Lagoon              Night Park     -0.388605   0.006829   
3  Sunway Lagoon                   Event      3.724688   0.075613   

        P_Value  Sample_Size  
0  3.463092e-09          631  
2  8.955440e-03          573  
1  4.629848e-02          582  
3  3.054070e-02           62  


In [17]:
def estimate_lost_world_ped(df):
    df = df.copy()
    
    # 1. Setup Date and filter for only Lost World
    df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date']).dt.date
    lw_df = df[df['Park'] == 'Lost World'].copy()
    
    # 2. Filter for Market Channels (Removing FOC and Membership)
    excluded = ['Foc / Complimentary', 'Membership', 'Staff']
    lw_df = lw_df[~lw_df['Grouping'].isin(excluded)]
    lw_df = lw_df[(lw_df['No_of_Visitors'] > 0) & (lw_df['Total_Revenue'] > 0)]
    
    # 3. Aggregation
    daily_agg = lw_df.groupby(['Transaction_Date', 'Grouping']).agg({
        'No_of_Visitors': 'sum',
        'Total_Revenue': 'sum'
    }).reset_index()
    daily_agg['Unit_Price'] = daily_agg['Total_Revenue'] / daily_agg['No_of_Visitors']
    
    # 4. Create the NEW Benchmark: 'Water Park'
    waterpark_ref = daily_agg[daily_agg['Grouping'] == 'Water Park'].copy()
    waterpark_ref = waterpark_ref.rename(columns={
        'Unit_Price': 'P_Base',
        'No_of_Visitors': 'Q_Base'
    })[['Transaction_Date', 'P_Base', 'Q_Base']]
    
    # 5. Join and Calculate Ratios
    merged = pd.merge(daily_agg, waterpark_ref, on='Transaction_Date')
    
    # Exclude the benchmark from the test rows
    processed = merged[merged['Grouping'] != 'Water Park'].copy()
    
    # Log Ratios
    processed['ln_Price_Ratio'] = np.log(processed['Unit_Price'] / processed['P_Base'])
    processed['ln_Qty_Ratio'] = np.log(processed['No_of_Visitors'] / processed['Q_Base'])
    
    # 6. Estimate PED
    results = []
    unique_channels = processed['Grouping'].unique()
    
    for channel in unique_channels:
        data = processed[processed['Grouping'] == channel].dropna()
        if len(data) < 15: continue # Sample size check
        
        X = sm.add_constant(data['ln_Price_Ratio'])
        y = data['ln_Qty_Ratio']
        model = sm.OLS(y, X).fit()
        
        results.append({
            'Park': 'Lost World',
            'Transaction_Channel': channel,
            'Relative_PED': model.params.iloc[1],
            'R_Squared': model.rsquared,
            'P_Value': model.pvalues.iloc[1],
            'Sample_Size': len(data)
        })
        
    return pd.DataFrame(results).sort_values(by='Relative_PED')

# Execute
lw_ped_results = estimate_lost_world_ped(df_channel_park)
print(lw_ped_results)

         Park   Transaction_Channel  Relative_PED  R_Squared       P_Value  \
0  Lost World        Group Function     -2.230259   0.134074  2.352501e-18   
1  Lost World             Hotspring     -1.074844   0.235287  2.262239e-39   
3  Lost World   Inter-Company/Staff     -0.870688   0.003579  6.830509e-01   
2  Lost World  Tours/Hotels/Presold     -0.381899   0.004237  1.804375e-01   

   Sample_Size  
0          533  
1          645  
3           49  
2          425  


In [18]:
def calculate_benchmark_volatility(df):
    df = df.copy()
    df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date']).dt.date
    
    # We define our benchmarks for each park
    benchmarks = [
        ('Sunway Lagoon', 'Walk In'),
        ('Lost World', 'Water Park')
    ]
    
    vol_results = []
    
    for park, channel in benchmarks:
        # Filter for the benchmark channel in each park
        data = df[(df['Park'] == park) & (df['Grouping'] == channel)]
        
        if data.empty:
            continue
            
        # Group by day to get daily total visitor counts
        daily_counts = data.groupby('Transaction_Date')['No_of_Visitors'].sum()
        
        # We use Coefficient of Variation (CV) = Standard Deviation / Mean
        # This allows us to compare "Stability" regardless of total volume
        mean_q = daily_counts.mean()
        std_q = daily_counts.std()
        cv = std_q / mean_q if mean_q > 0 else 0
        
        # Assign a recommended benchmark based on CV
        if cv > 1.0:
            rec_ped = -1.2 # Highly volatile/fickle
        elif cv > 0.6:
            rec_ped = -0.8 # Moderate/Standard
        else:
            rec_ped = -0.5 # Very stable/steady
            
        vol_results.append({
            'Park': park,
            'Benchmark_Channel': channel,
            'Mean_Daily_Visitors': round(mean_q, 2),
            'Volatility_Index_(CV)': round(cv, 4),
            'Recommended_Benchmark_PED': rec_ped
        })
        
    return pd.DataFrame(vol_results)

# --- Execute ---
volatility_table = calculate_benchmark_volatility(df_channel_park)
print(volatility_table)

            Park Benchmark_Channel  Mean_Daily_Visitors  \
0  Sunway Lagoon           Walk In               380.30   
1     Lost World        Water Park               236.17   

   Volatility_Index_(CV)  Recommended_Benchmark_PED  
0                 0.4669                       -0.5  
1                 0.9751                       -0.8  
